# FORMATION-MUX-001 exact-resume recovery — Kaggle T4 ×2

Use this notebook only to recover the pinned 2026-09-23 execution. Attach the complete saved Kaggle Output tree, not only `FORMATION_MUX_001_RESULTS.zip`.

Cell 1 verifies science/data/tokenizer identities, the 24 completed S5 arms, required completed T0 controls, every official checkpoint receipt hash, and sealed-marker consistency. It refuses evidence-only ZIPs and never overwrites an existing working campaign.

Cell 2 runs only the pinned operator v12 at commit `4ee05f6e386f15d34f9dfa7bd7f3300a496b9896`. Completed arms remain immutable; no seeds, arms, exposures, endpoints, thresholds, or sealed policy are changed.

If Cell 1 fails, stop and preserve the attached Output. Do not bypass the preflight or relabel a rerun as recovery.


In [ ]:
import pathlib
import subprocess
import sys

REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
RECOVERY_COMMIT = '7a82d80e6f44756c9e37f3b554500f4f5c664238'
RECOVERY_BLOB = 'b9274f974e47aab9fc272c924b519b181d21154a'
RECOVERY_REPO = pathlib.Path('/kaggle/temp/formation-mux-recovery-' + RECOVERY_COMMIT[:12])
OUTPUT = pathlib.Path('/kaggle/working/FORMATION_MUX_001')

if RECOVERY_REPO.exists():
    if not (RECOVERY_REPO / '.git').exists():
        raise RuntimeError(f'Refusing to overwrite non-Git recovery path: {RECOVERY_REPO}')
    status = subprocess.run(['git', '-C', str(RECOVERY_REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if status.strip():
        raise RuntimeError(f'Refusing to modify dirty recovery checkout: {RECOVERY_REPO}')
    head = subprocess.run(['git', '-C', str(RECOVERY_REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
    if head != RECOVERY_COMMIT:
        raise RuntimeError(f'Refusing to switch recovery checkout from {head}')
else:
    subprocess.run(['git', 'clone', '--no-checkout', REMOTE, str(RECOVERY_REPO)], check=True)
    subprocess.run(['git', '-C', str(RECOVERY_REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(RECOVERY_REPO), 'switch', '--detach', RECOVERY_COMMIT], check=True)

recovery_blob = subprocess.run(['git', '-C', str(RECOVERY_REPO), 'hash-object', 'tools/formation_mux_001_recovery_preflight.py'], capture_output=True, text=True, check=True).stdout.strip()
assert recovery_blob == RECOVERY_BLOB, (recovery_blob, RECOVERY_BLOB)
subprocess.run([sys.executable, '-m', 'tools.formation_mux_001_recovery_preflight', '--input', '/kaggle/input', '--out', str(OUTPUT), '--install'], cwd=str(RECOVERY_REPO), check=True)
print('RECOVERY CUSTODY: PASS')


In [ ]:
import json
import pathlib
import subprocess
import sys

OPERATOR_COMMIT = '4ee05f6e386f15d34f9dfa7bd7f3300a496b9896'
OPERATOR_BLOB = 'e9e1f701b0d4edc509194da55fe1ba37ed62ef86'
SCIENCE_COMMIT = 'c15ad8beb409537db42d075684ea54847a074ebd'
OPERATOR_REPO = pathlib.Path('/kaggle/temp/formation-mux-source-' + OPERATOR_COMMIT[:12])
OUTPUT = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'

if OPERATOR_REPO.exists():
    if not (OPERATOR_REPO / '.git').exists():
        raise RuntimeError(f'Refusing to overwrite non-Git operator path: {OPERATOR_REPO}')
    status = subprocess.run(['git', '-C', str(OPERATOR_REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if status.strip():
        raise RuntimeError(f'Refusing to modify dirty operator checkout: {OPERATOR_REPO}')
    head = subprocess.run(['git', '-C', str(OPERATOR_REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
    if head != OPERATOR_COMMIT:
        raise RuntimeError(f'Refusing to switch operator checkout from {head}')
else:
    subprocess.run(['git', 'clone', '--no-checkout', REMOTE, str(OPERATOR_REPO)], check=True)
    subprocess.run(['git', '-C', str(OPERATOR_REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(OPERATOR_REPO), 'switch', '--detach', OPERATOR_COMMIT], check=True)

head = subprocess.run(['git', '-C', str(OPERATOR_REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
blob = subprocess.run(['git', '-C', str(OPERATOR_REPO), 'hash-object', 'tools/formation_mux_001_kaggle_operator_v12.py'], capture_output=True, text=True, check=True).stdout.strip()
assert head == OPERATOR_COMMIT, (head, OPERATOR_COMMIT)
assert blob == OPERATOR_BLOB, (blob, OPERATOR_BLOB)
subprocess.run(['git', '-C', str(OPERATOR_REPO), 'cat-file', '-e', SCIENCE_COMMIT + '^{commit}'], check=True)
if not (OUTPUT / 'RECOVERY_PREFLIGHT.json').exists():
    raise RuntimeError('RECOVERY_PREFLIGHT.json missing; refusing operator launch')
recovery = json.loads((OUTPUT / 'RECOVERY_PREFLIGHT.json').read_text())
if recovery.get('status') != 'PASS':
    raise RuntimeError('recovery preflight is not PASS; refusing operator launch')

completed = subprocess.run([sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v12.py', '--repo', str(OPERATOR_REPO), '--out', str(OUTPUT)], cwd=str(OPERATOR_REPO))
if completed.returncode != 0:
    for name in ('GLOBAL_FAILURE_V12.json', 'GLOBAL_FAILURE_V10.json', 'GLOBAL_FAILURE.json'):
        failure = OUTPUT / name
        if failure.exists():
            print(failure.read_text())
    raise RuntimeError('pinned operator failed closed; preserve the entire working and saved Output trees')

s5 = json.loads((OUTPUT / 'CAMPAIGN_STATE.json').read_text())
frontier = json.loads((OUTPUT / 'TIE_ROLE_FRONTIER_STATE.json').read_text())
print('S5:', s5.get('status'), s5.get('complete_arms'), '/', s5.get('required_arms'))
print('FRONTIER:', frontier.get('status'), frontier.get('complete_arms'), '/', frontier.get('required_arms'))
for experiment in ('CS-MECH-002', 'REP-FORM-003A', 'TIE-ROLE-001', 'TIE-ROLE-XFER-001'):
    final = OUTPUT / experiment / 'FINAL_RESULT.json'
    print(experiment, 'FINAL_RESULT', final.exists())
print('Save this Kaggle version and preserve the complete /kaggle/working/FORMATION_MUX_001 tree before another session.')
